# CodeCanvas: Vulnerability Hotspot Detector (CodeBERT)
This notebook fine-tunes **microsoft/codebert-base** on a vulnerability dataset to detect security hotspots semantically.

In [ ]:
!pip install transformers datasets torch scikit-learn

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

In [ ]:
print("Loading CodeBERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained('microsoft/codebert-base')
model = AutoModelForSequenceClassification.from_pretrained('microsoft/codebert-base', num_labels=2)

In [ ]:
print('Downloading Devign/CodeXGLUE defect dataset...')
dataset = load_dataset('google/code_x_glue_cc_defect_detection')

# Downsample for speed (so training takes a few minutes instead of hours for the demo)
train_dataset = dataset['train'].shuffle(seed=42).select(range(1000))
eval_dataset = dataset['validation'].shuffle(seed=42).select(range(200))

In [ ]:
def tokenize_function(examples):
    # Max length 256 for faster processing on Colab
    tokenized = tokenizer(examples['func'], padding='max_length', truncation=True, max_length=256)
    # The Trainer expects the label column to be named 'labels'
    tokenized['labels'] = examples['target']
    return tokenized

print("Tokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    probs = torch.nn.functional.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    try:
        roc_auc = roc_auc_score(labels, probs)
    except:
        roc_auc = 0.5
    
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall, 'roc_auc': roc_auc}

In [ ]:
print("Configuring Trainer...")
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print("Starting Training...")
trainer.train()
print('✅ Training complete! Take your screenshots now.')